In [ ]:
import uuid
from os import getenv
from typing import Callable
from uuid import uuid4

from langchain_core.runnables import Runnable, RunnableConfig, RunnableLambda

from agents import interpreter_agent
from utils import get_text_util
from utils.run_with_hitl_util import run_with_hitl
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
from utils.session_util import ResolvedSession

In [ ]:
agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite"),
)
session = ResolvedSession(agent=agent, thread_id="new", mode_name="custom", is_secret=False)

In [ ]:
res = run_with_hitl(
    initial_input={"messages" : [HumanMessage(content="hello there!")]},
    agent=session.agent,
    config=session.config,
)

In [ ]:
from utils.get_text_util import get_text
def adapter(state: dict):
    message = get_text(state["messages"][-1])
    config = session.config
    res = interpreter_agent.invoke({"messages": [HumanMessage(content=message)]}, config)
    return AIMessage(get_text(res))


In [1]:
DEFAULT_SUBAGENT_NAME = "code_interpreter"
DEFAULT_SUBAGENT_DESCRIPTION = (
        """Executes Python scripts, computational tasks, data processing, and shell commands in an isolated sandbox workspace. Generates deliverable files (CSVs, charts markdown reports) in artifacts/ and self-corrects on errors."""
    )

In [3]:
from pathlib import Path
from utils.sandbox_util import list_sandbox_artifacts

def make_interpreter_subagent(
        agent: Runnable,
        name: str = DEFAULT_SUBAGENT_NAME,
        description: str = DEFAULT_SUBAGENT_DESCRIPTION,
        sandbox_dir: Path | None = None,
):
    def _run_interpreter_subagent(state: dict):

        task_text = state["messages"][-1]
        thread_id = str(uuid.uuid4())
        config:RunnableConfig = {"configurable":{"thread_id":thread_id}}

        prev_arts = {a.path: (a.size_bytes, a.modified_at) for a in list_sandbox_artifacts()}
        result = agent.invoke({"messages": [HumanMessage(content=task_text)]}, config=config)
        output_text = get_text(result["messages"][-1])

        current_artifacts = list_sandbox_artifacts(sandbox_dir=sandbox_dir)
        new_arts = [a for a in current_artifacts if a.path not in prev_arts]
        modified_arts = []

        for a in current_artifacts:
            if a.path in prev_arts:
                prev_size, prev_mtime = prev_arts[a.path]
                has_size_changed = a.size_bytes != prev_size
                has_time_changed = (
                    a.modified_at is not None
                    and prev_mtime is not None
                    and a.modified_at > prev_mtime
                )
                if has_size_changed or has_time_changed:
                    modified_arts.append(a)

        parts = [output_text]
        if new_arts or modified_arts:
            for artifact in new_arts:
                size_kb = f" ({artifact.size_bytes / 1024:.1f} KB)" if artifact.size_bytes else ""
                desc = f" - {artifact.description}" if artifact.description else ""
                parts.append(f"- `[NEW]` `{artifact.path}`{size_kb} [{artifact.artifact_type}]{desc}")

            for artifact in modified_arts:
                size_kb = f" ({artifact.size_bytes / 1024:.1f} KB)" if artifact.size_bytes else ""
                desc = f" - {artifact.description}" if artifact.description else ""
                parts.append(f"- `[MODIFIED]` `{artifact.path}`{size_kb}[{artifact.artifact_type}]{desc}")

        return {
            "messages": AIMessage(content="\n".join(parts)),
            "thread_id": thread_id,
        }

    return {
        "name": name,
        "description": description,
        "runnable": RunnableLambda(_run_interpreter_subagent),
    }




In [5]:
from openai import OpenAI
import os

# ایجاد یک نمونه از کلاینت با کلید API خود
client = OpenAI(base_url='https://api.gapgpt.app/v1', api_key=os.getenv("GAPGPT_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "سلام!"}
    ]
)

print(response.choices[0].message.content)


AuthenticationError: Error code: 401 - {'error': {'code': '', 'message': 'Invalid token (request id: 202609090032397621860528268d9d6H34jLmZ9)', 'type': 'api_provider_error'}}